# 🎯 Veri Madenciliği — Sınıflandırma: Temel Kavramlar ve Teknikler
**Haydar Kılıç | Mühendislik Fakültesi, Yapay Zeka Mühendisliği | Bahar Yarıyılı**

---

Bu notebook, ders sunumundaki teorik kavramların Python ile uygulamalı gösterimidir.

## 📋 İçindekiler
1. [Sınıflandırma Nedir?](#1)
2. [Omurgalı Veri Seti Örneği](#2)
3. [Confusion Matrix ve Performans Metrikleri](#3)
4. [Karar Ağacı Sınıflandırması](#4)
5. [Hunt Algoritması Simülasyonu](#5)
6. [Öznitelik Türleri ve Bölme Koşulları](#6)
7. [Safsızlık Ölçüleri: Entropi, Gini, Sınıflandırma Hatası](#7)
8. [Bilgi Kazancı ve Kazanç Oranı](#8)
9. [Overfitting ve Underfitting](#9)
10. [Model Seçimi ve Cross-Validation](#10)
11. [Model Karşılaştırması — İstatistiksel Test](#11)

## ⚙️ Kütüphaneler

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings('ignore')

from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import confusion_matrix, accuracy_score, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11
print("✅ Tüm kütüphaneler başarıyla yüklendi!")

---
<a id='1'></a>
## 1. Sınıflandırma Nedir?

**Tanım:** Her kayıt bir `(x, y)` ikilisiyle temsil edilir:
- **x** → öznitelik kümesi (bağımsız değişken, girdi)
- **y** → sınıf etiketi (bağımlı değişken, çıktı)

**Görev:** Her `x` için önceden tanımlanmış bir `y` etiketi tahmin eden model öğrenmek.

### Gerçek Dünyadan Örnekler

In [ ]:
gorevler = pd.DataFrame({
    'Görev': ['Spam Filtreleme', 'Tümör Tanımlama', 'Galaksi Sınıflandırma', 'Kredi Riski'],
    'Öznitelik Kümesi (x)': [
        'E-posta başlığı/içeriği özellikleri',
        'MRI tarama görüntüsü özellikleri',
        'Teleskop görüntüsü özellikleri',
        'Gelir, medeni durum, ev sahipliği'
    ],
    'Sınıf Etiketi (y)': [
        'Spam / Spam Değil',
        'Malign / Benign',
        'Eliptik / Spiral / Düzensiz',
        'Temerrüt / Temerrüt Yok'
    ]
})

print("📊 Sınıflandırma Görevlerine Örnekler")
print("=" * 80)
print(gorevler.to_string(index=False))

### Tümevarım (Induction) ve Tümdengelim (Deduction)

- **Tümevarım:** Eğitim verisinden model öğrenme süreci → *"Model Öğrenme"*
- **Tümdengelim:** Öğrenilen modeli yeni örneklere uygulama → *"Modeli Uygulama"*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
np.random.seed(42)
X_demo = np.random.randn(50, 2)
y_demo = (X_demo[:, 0] + X_demo[:, 1] > 0).astype(int)
colors = ['#e74c3c' if y == 1 else '#3498db' for y in y_demo]
ax1.scatter(X_demo[:, 0], X_demo[:, 1], c=colors, s=80, alpha=0.8,
            edgecolors='white', linewidth=0.5)
ax1.set_title('📚 Eğitim Kümesi (Tümevarım)\n(x, y) ikililerinden model öğrenme',
              fontsize=12, fontweight='bold')
ax1.set_xlabel('x₁ (Öznitelik 1)')
ax1.set_ylabel('x₂ (Öznitelik 2)')
red_patch = mpatches.Patch(color='#e74c3c', label='Sınıf 1')
blue_patch = mpatches.Patch(color='#3498db', label='Sınıf 0')
ax1.legend(handles=[red_patch, blue_patch])
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
X_test_demo = np.random.randn(20, 2)
clf_demo = DecisionTreeClassifier(max_depth=3, random_state=42)
clf_demo.fit(X_demo, y_demo)
y_pred_demo = clf_demo.predict(X_test_demo)

xx, yy = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
Z = clf_demo.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax2.contourf(xx, yy, Z, alpha=0.2, cmap=ListedColormap(['#3498db', '#e74c3c']))
ax2.contour(xx, yy, Z, colors='black', linewidths=1.5, linestyles='--')

test_colors = ['#e74c3c' if y == 1 else '#3498db' for y in y_pred_demo]
ax2.scatter(X_test_demo[:, 0], X_test_demo[:, 1], c=test_colors, s=100,
            alpha=0.9, edgecolors='black', linewidth=1.5, marker='*', zorder=5)
ax2.set_title('🔍 Test Kümesi (Tümdengelim)\nÖğrenilen model ile tahmin yapma',
              fontsize=12, fontweight='bold')
ax2.set_xlabel('x₁ (Öznitelik 1)')
ax2.set_ylabel('x₂ (Öznitelik 2)')
ax2.legend(handles=[red_patch, blue_patch])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle('Sınıflandırma Süreci: Tümevarım → Tümdengelim',
             fontsize=13, fontweight='bold', y=1.02)
plt.show()

---
<a id='2'></a>
## 2. Omurgalı Veri Seti Örneği

Sunumdaki omurgalı veri setini Python'da oluşturup işleyeceğiz.

In [ ]:
vertebrate_data = {
    'Isim': ['human', 'python', 'salmon', 'whale', 'frog',
             'komodo dragon', 'bat', 'pigeon', 'cat'],
    'Vucut_Sicakligi': ['warm', 'cold', 'cold', 'warm', 'cold',
                         'cold', 'warm', 'warm', 'warm'],
    'Deri_Ortusu': ['hair', 'scales', 'scales', 'hair', 'none',
                    'scales', 'hair', 'feathers', 'fur'],
    'Dogum': [True, False, False, True, False, False, True, False, True],
    'Sucul': [False, False, True, True, True, False, False, False, False],
    'Havaci': [False, False, False, False, False, False, True, True, False],
    'Bacak': [True, False, False, False, True, True, True, True, True],
    'Kis_Uykusu': [False, True, False, False, True, False, True, False, False],
    'Sinif': ['mammal', 'reptile', 'fish', 'mammal', 'amphibian',
              'reptile', 'mammal', 'bird', 'mammal']
}

df_vertebrate = pd.DataFrame(vertebrate_data)
print('🦎 Omurgalı Veri Seti')
print('=' * 80)
print(df_vertebrate.to_string(index=False))
print(f'\n📊 Veri boyutu: {df_vertebrate.shape[0]} örnek, {df_vertebrate.shape[1]} sütun')
print(f"🏷️  Sınıflar: {df_vertebrate['Sinif'].unique()}")

In [ ]:
df_vertebrate['Sinif_Ikili'] = df_vertebrate['Sinif'].apply(
    lambda x: 'Memeli' if x == 'mammal' else 'Memeli Degil'
)

print('🔵 İkili Sınıflandırma Dönüşümü (Memeli / Memeli Değil)')
print('=' * 60)
print(df_vertebrate[['Isim', 'Sinif', 'Sinif_Ikili']].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sinif_sayilari = df_vertebrate['Sinif'].value_counts()
colors_pie = plt.cm.Set3(np.linspace(0, 1, len(sinif_sayilari)))
axes[0].pie(sinif_sayilari.values, labels=sinif_sayilari.index,
            colors=colors_pie, autopct='%1.0f%%', startangle=90)
axes[0].set_title('Çok Sınıflı Dağılım', fontweight='bold')

ikili_sayilari = df_vertebrate['Sinif_Ikili'].value_counts()
axes[1].bar(ikili_sayilari.index, ikili_sayilari.values,
            color=['#2ecc71', '#e74c3c'], edgecolor='white', linewidth=1.5)
axes[1].set_title('İkili Sınıflandırma Dağılımı', fontweight='bold')
axes[1].set_ylabel('Örnek Sayısı')
for i, (label, val) in enumerate(zip(ikili_sayilari.index, ikili_sayilari.values)):
    axes[1].text(i, val + 0.05, str(val), ha='center', fontweight='bold', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
print('🦎 Test Örneği: Gila Monster')
print('=' * 50)
print('Vücut Sıcaklığı : cold')
print('Deri Örtüsü     : scales')
print('Doğum Yapıyor   : Hayır')
print('Sucul Canlı     : Hayır')
print('Havacı          : Hayır')
print('Bacağı Var      : Evet')
print('Kış Uykusu      : Evet')

def karar_agaci_tahmin(vucut_sicakligi, dogum_yapiyor):
    """Sunumdaki basit karar ağacını simüle eder."""
    if vucut_sicakligi == 'warm':
        return 'Memeli' if dogum_yapiyor else 'Memeli Degil'
    return 'Memeli Degil'

tahmin = karar_agaci_tahmin('cold', False)
print(f'\n✅ Karar Ağacı Tahmini → {tahmin}')
print('   Yol: Vücut Sıcaklığı=cold → Yaprak: Memeli Değil')

---
<a id='3'></a>
## 3. Confusion Matrix (Hata Matrisi) ve Performans Metrikleri

$$\\text{Doğruluk} = \\frac{f_{11} + f_{00}}{f_{11} + f_{10} + f_{01} + f_{00}}$$

$$\\text{Hata Oranı} = \\frac{f_{10} + f_{01}}{f_{11} + f_{10} + f_{01} + f_{00}}$$

In [ ]:
le = LabelEncoder()
X_vt = pd.get_dummies(df_vertebrate[['Vucut_Sicakligi', 'Deri_Ortusu',
                                      'Dogum', 'Sucul', 'Havaci',
                                      'Bacak', 'Kis_Uykusu']])
y_vt = le.fit_transform(df_vertebrate['Sinif_Ikili'])

clf_vt = DecisionTreeClassifier(max_depth=3, random_state=42)
clf_vt.fit(X_vt, y_vt)
y_pred_vt = clf_vt.predict(X_vt)

cm = confusion_matrix(y_vt, y_pred_vt)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix (Hata Matrisi)', fontweight='bold', fontsize=12)

f11, f10, f01, f00 = cm[1,1], cm[1,0], cm[0,1], cm[0,0]
toplam = f11 + f10 + f01 + f00
dogruluk = (f11 + f00) / toplam
hata_orani = (f10 + f01) / toplam

metriks = ['f₁₁ (TP)', 'f₀₀ (TN)', 'f₁₀ (FN)', 'f₀₁ (FP)']
degerler = [f11, f00, f10, f01]
bar_colors = ['#2ecc71', '#2ecc71', '#e74c3c', '#e74c3c']
bars = axes[1].bar(metriks, degerler, color=bar_colors, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, degerler):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                str(val), ha='center', va='bottom', fontweight='bold', fontsize=13)
axes[1].set_title('Confusion Matrix Değerleri', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Örnek Sayısı')
axes[1].set_ylim(0, max(degerler) + 0.8)

plt.tight_layout()
plt.show()

print(f'\n📊 Performans Metrikleri')
print(f'{"="*40}')
print(f'  f₁₁ (Gerçek Pozitif - TP) : {f11}')
print(f'  f₀₀ (Gerçek Negatif - TN) : {f00}')
print(f'  f₁₀ (Yanlış Negatif - FN) : {f10}')
print(f'  f₀₁ (Yanlış Pozitif - FP) : {f01}')
print(f'  Toplam örnek sayısı        : {toplam}')
print()
print(f'  ✅ Doğruluk (Accuracy)  = ({f11}+{f00})/{toplam} = {dogruluk:.3f} = %{dogruluk*100:.1f}')
print(f'  ❌ Hata Oranı (Error)   = ({f10}+{f01})/{toplam} = {hata_orani:.3f} = %{hata_orani*100:.1f}')

---
<a id='4'></a>
## 4. Karar Ağacı Yapısı ve Sınıflandırma

### Karar Ağacı Bileşenleri
- **Kök Düğüm (Root Node):** Gelen bağlantısı olmayan başlangıç düğümü
- **İç Düğümler (Internal Nodes):** Nitelik test koşulları içeren düğümler
- **Yaprak/Uç Düğümler (Leaf/Terminal Nodes):** Sınıf etiketiyle ilişkili bitiş düğümleri

In [ ]:
kredit_data = {
    'ID': range(1, 11),
    'Ev_Sahibi': ['Yes','No','No','Yes','No','No','Yes','No','No','No'],
    'Medeni_Durum': ['Single','Married','Single','Married','Divorced',
                     'Married','Divorced','Single','Married','Single'],
    'Yillik_Gelir': [125000, 100000, 70000, 120000, 95000,
                     60000, 220000, 85000, 75000, 90000],
    'Temerrut': ['No','No','No','No','Yes','No','No','Yes','No','Yes']
}
df_kredit = pd.DataFrame(kredit_data)
print('💰 Kredi Borçlusu Veri Seti (Hunt Algoritması Örneği)')
print('=' * 65)
print(df_kredit.to_string(index=False))
print(f'\n📊 Temerrüt Dağılımı:')
print(df_kredit['Temerrut'].value_counts().to_string())

In [ ]:
le2 = LabelEncoder()
df_kredit_enc = df_kredit.copy()
df_kredit_enc['Ev_Sahibi_enc'] = le2.fit_transform(df_kredit['Ev_Sahibi'])
df_kredit_enc['Medeni_Durum_enc'] = le2.fit_transform(df_kredit['Medeni_Durum'])
df_kredit_enc['Temerrut_enc'] = le2.fit_transform(df_kredit['Temerrut'])

X_kredit = df_kredit_enc[['Ev_Sahibi_enc', 'Medeni_Durum_enc', 'Yillik_Gelir']]
y_kredit = df_kredit_enc['Temerrut_enc']

clf_kredit = DecisionTreeClassifier(max_depth=4, criterion='entropy', random_state=42)
clf_kredit.fit(X_kredit, y_kredit)

fig, ax = plt.subplots(figsize=(16, 7))
plot_tree(clf_kredit,
          feature_names=['Ev Sahibi', 'Medeni Durum', 'Yıllık Gelir'],
          class_names=['Temerrüt=Hayır', 'Temerrüt=Evet'],
          filled=True, rounded=True, fontsize=10, ax=ax,
          impurity=True, precision=3)
ax.set_title('💰 Kredi Borçlusu Karar Ağacı\n(Entropi Kriteri, Max Derinlik=4)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n🌳 Ağaç Yapısı (Metin Formatı):')
print('=' * 50)
print(export_text(clf_kredit,
                  feature_names=['Ev_Sahibi', 'Medeni_Durum', 'Yillik_Gelir']))

---
<a id='5'></a>
## 5. Hunt Algoritması Simülasyonu

Hunt algoritması, karar ağacını **özyinelemeli (recursive)** olarak büyütür:
1. Tüm örnekler aynı sınıftaysa → yaprak düğüm oluştur
2. Değilse → en iyi bölme kriterine göre genişlet
3. Her çocuk düğüme özyinelemeli uygula

In [ ]:
def hesapla_entropi(y):
    """Bir düğümdeki entropi değerini hesaplar."""
    if len(y) == 0:
        return 0
    siniflar, sayilar = np.unique(y, return_counts=True)
    oranlar = sayilar / len(y)
    return -np.sum(oranlar * np.log2(oranlar + 1e-10))


def hunt_adim(veri, hedef_sutun, seviye=0, max_seviye=3):
    """Hunt algoritmasının özyinelemeli simülasyonu."""
    girintili = '  ' * seviye
    siniflar = veri[hedef_sutun].unique()
    entropi = hesapla_entropi(veri[hedef_sutun])

    print(f'{girintili}📦 Düğüm | Örnek sayısı: {len(veri)} | '
          f'Entropi: {entropi:.3f} | Dağılım: {dict(veri[hedef_sutun].value_counts())}')

    if len(siniflar) == 1:
        print(f'{girintili}  ✅ YAPRAK → Sınıf: {siniflar[0]}')
        return

    if seviye >= max_seviye:
        cok_sinif = veri[hedef_sutun].value_counts().idxmax()
        print(f'{girintili}  🛑 Max derinlik → Çoğunluk sınıfı: {cok_sinif}')
        return

    en_iyi_kazanc = -1
    en_iyi_oznitelik = None

    for sutun in ['Ev_Sahibi', 'Medeni_Durum']:
        kazanc = entropi
        for deger in veri[sutun].unique():
            alt_set = veri[veri[sutun] == deger]
            agirlik = len(alt_set) / len(veri)
            kazanc -= agirlik * hesapla_entropi(alt_set[hedef_sutun])
        if kazanc > en_iyi_kazanc:
            en_iyi_kazanc = kazanc
            en_iyi_oznitelik = sutun

    esik = 78000
    kazanc_gelir = entropi
    for maske in [veri['Yillik_Gelir'] < esik, veri['Yillik_Gelir'] >= esik]:
        alt_set = veri[maske]
        if len(alt_set) > 0:
            agirlik = len(alt_set) / len(veri)
            kazanc_gelir -= agirlik * hesapla_entropi(alt_set[hedef_sutun])
    if kazanc_gelir > en_iyi_kazanc:
        en_iyi_kazanc = kazanc_gelir
        en_iyi_oznitelik = f'Yillik_Gelir<{esik}'

    print(f'{girintili}  🔀 En iyi bölme: [{en_iyi_oznitelik}] | Bilgi Kazancı: {en_iyi_kazanc:.3f}')

    if en_iyi_oznitelik and '<' not in en_iyi_oznitelik:
        for deger in sorted(veri[en_iyi_oznitelik].unique()):
            alt_set = veri[veri[en_iyi_oznitelik] == deger]
            print(f'{girintili}  ↳ {en_iyi_oznitelik} = {deger}')
            hunt_adim(alt_set, hedef_sutun, seviye + 1, max_seviye)
    else:
        evet_set = veri[veri['Yillik_Gelir'] < esik]
        hayir_set = veri[veri['Yillik_Gelir'] >= esik]
        print(f'{girintili}  ↳ Yillik_Gelir < {esik}')
        hunt_adim(evet_set, hedef_sutun, seviye + 1, max_seviye)
        print(f'{girintili}  ↳ Yillik_Gelir >= {esik}')
        hunt_adim(hayir_set, hedef_sutun, seviye + 1, max_seviye)


print('🌳 Hunt Algoritması Simülasyonu — Kredi Borçlusu Veri Seti')
print('=' * 65)
hunt_adim(df_kredit, 'Temerrut', max_seviye=2)

---
<a id='6'></a>
## 6. Öznitelik Türleri ve Bölme Koşulları

| Tür | Açıklama | Bölme Yöntemi |
|-----|----------|---------------|
| **Nominal** | Sırasız kategorik (örn. renk, medeni durum) | Çok yollu veya ikili |
| **Sıralı (Ordinal)** | Sıralı kategorik (örn. boy: küçük/orta/büyük) | Sıralamayı koruyarak |
| **Sürekli (Continuous)** | Sayısal (örn. gelir, yaş) | Eşik değeriyle ikili / aralık sorgusu |

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Nominal: Medeni Durum — Çok yollu bölme
ax = axes[0]
kategoriler = ['Bekar', 'Evli', 'Boşanmış']
temerrut_orani = [0.40, 0.00, 0.50]
bars = ax.bar(kategoriler, temerrut_orani,
               color=['#3498db', '#2ecc71', '#e74c3c'],
               edgecolor='white', linewidth=2)
for bar, val in zip(bars, temerrut_orani):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'%{val*100:.0f}', ha='center', fontweight='bold')
ax.set_title('📊 Nominal: Medeni Durum\n(Çok Yollu Bölme)', fontweight='bold')
ax.set_ylabel('Temerrüt Oranı')
ax.set_ylim(0, 0.7)
ax.axhline(y=0.3, color='gray', linestyle='--', alpha=0.5, label='Genel oran')
ax.legend()

# 2. Sıralı: Gömlek Bedeni — İkili bölme
ax = axes[1]
bedenler = ['S', 'M', 'L', 'XL']
renkler_beden = ['#3498db', '#3498db', '#e74c3c', '#e74c3c']
bars2 = ax.bar(bedenler, [3, 4, 5, 6], color=renkler_beden,
                edgecolor='white', linewidth=2)
ax.axvline(x=1.5, color='black', linestyle='--', linewidth=2.5, label='Bölme noktası')
ax.text(0.5, 6.2, '{S,M}', ha='center', color='#3498db', fontweight='bold', fontsize=12)
ax.text(2.5, 6.2, '{L,XL}', ha='center', color='#e74c3c', fontweight='bold', fontsize=12)
ax.set_title('📏 Sıralı: Gömlek Bedeni\n(Sıralama Korunarak İkili Bölme)', fontweight='bold')
ax.set_ylabel('Örnek Sayısı')
ax.legend()
ax.set_ylim(0, 7.5)

# 3. Sürekli: Yıllık Gelir
ax = axes[2]
np.random.seed(42)
gelir_no = np.random.normal(55000, 20000, 30)
gelir_yes = np.random.normal(95000, 25000, 15)
ax.hist(gelir_no, bins=10, alpha=0.6, color='#3498db', label='Temerrüt=Hayır', edgecolor='white')
ax.hist(gelir_yes, bins=10, alpha=0.6, color='#e74c3c', label='Temerrüt=Evet', edgecolor='white')
ax.axvline(x=78000, color='black', linestyle='--', linewidth=2.5, label='Eşik: 78K')
ax.set_title('💵 Sürekli: Yıllık Gelir\n(Eşik Değeri ile İkili Bölme)', fontweight='bold')
ax.set_xlabel('Yıllık Gelir ($)')
ax.set_ylabel('Frekans')
ax.legend(fontsize=9)

plt.suptitle('Öznitelik Türleri ve Bölme Koşulları', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\n💡 Özet:')
print('  • Nominal nitelikler → Çok yollu veya ikili bölme')
print('  • Sıralı nitelikler → Sıralama özelliği bozulmadan gruplandırma')
print('  • Sürekli nitelikler → A < v (ikili) veya vi ≤ A < vi+1 (çok yollu aralık)')

---
<a id='7'></a>
## 7. Safsızlık Ölçüleri: Entropi, Gini İndeksi, Sınıflandırma Hatası

$$\\text{Entropi} = -\\sum_{i=0}^{c-1} p_i(t) \\log_2 p_i(t)$$

$$\\text{Gini} = 1 - \\sum_{i=0}^{c-1} p_i(t)^2$$

$$\\text{Sınıflandırma Hatası} = 1 - \\max_i[p_i(t)]$$

In [ ]:
def entropi(p):
    """İkili sınıflandırma için entropi. p = P(Sınıf=1)"""
    if p == 0 or p == 1:
        return 0.0
    return -p * np.log2(p) - (1 - p) * np.log2(1 - p)

def gini(p):
    """İkili sınıflandırma için Gini indeksi."""
    return 1 - p**2 - (1 - p)**2

def siniflandirma_hatasi(p):
    """Sınıflandırma hatası."""
    return 1 - max(p, 1 - p)

p_values = np.linspace(0, 1, 500)
entropi_vals = [entropi(p) for p in p_values]
gini_vals = [gini(p) for p in p_values]
hata_vals = [siniflandirma_hatasi(p) for p in p_values]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(p_values, entropi_vals, 'b-', linewidth=2.5, label='Entropi')
axes[0].plot(p_values, gini_vals, 'r--', linewidth=2.5, label='Gini İndeksi')
axes[0].plot(p_values, hata_vals, 'g-.', linewidth=2.5, label='Sınıflandırma Hatası')
axes[0].axvline(x=0.5, color='gray', linestyle=':', alpha=0.7, label='p=0.5 (max safsızlık)')
axes[0].set_xlabel('p (Sınıf 1 oranı)', fontsize=12)
axes[0].set_ylabel('Safsızlık Değeri', fontsize=12)
axes[0].set_title('Safsızlık Ölçütleri Karşılaştırması', fontweight='bold', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

dugumler = ['N1\n(0/6,6/6)', 'N2\n(1/6,5/6)', 'N3\n(3/6,3/6)']
p_vals_orn = [1.0, 5/6, 0.5]
e_vals = [entropi(p) for p in p_vals_orn]
g_vals = [gini(p) for p in p_vals_orn]
h_vals = [siniflandirma_hatasi(p) for p in p_vals_orn]

x = np.arange(len(dugumler))
w = 0.25
axes[1].bar(x - w, e_vals, w, label='Entropi', color='#3498db', edgecolor='white')
axes[1].bar(x,     g_vals, w, label='Gini',    color='#e74c3c', edgecolor='white')
axes[1].bar(x + w, h_vals, w, label='Hata',    color='#2ecc71', edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_xticklabels(dugumler, fontsize=10)
axes[1].set_ylabel('Safsızlık Değeri')
axes[1].set_title('Düğüm Örnekleri (N1, N2, N3)', fontweight='bold', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')

for container in axes[1].containers:
    axes[1].bar_label(container, fmt='%.3f', fontsize=8, padding=2)

plt.suptitle('Safsızlık Ölçüleri: Entropi, Gini, Sınıflandırma Hatası',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print('📐 Safsızlık Ölçüsü Hesaplamaları (Sunumdaki Örnekler)')
print('=' * 60)

dugumler_data = [
    ('N1', 0, 6),
    ('N2', 1, 5),
    ('N3', 3, 3),
]

for isim, c0, c1 in dugumler_data:
    toplam = c0 + c1
    p0, p1 = c0 / toplam, c1 / toplam
    p = p1

    gini_val = 1 - p0**2 - p1**2
    ent_val  = entropi(p)
    err_val  = siniflandirma_hatasi(p)

    print(f'\n{"─"*50}')
    print(f'  Düğüm {isim}: Sınıf=0: {c0} örnek, Sınıf=1: {c1} örnek')
    print(f'  p₀={p0:.4f}, p₁={p1:.4f}')
    print(f'  Gini    = 1 - ({p0:.3f})² - ({p1:.3f})² = {gini_val:.3f}')
    print(f'  Entropi = {ent_val:.3f}')
    print(f'  Hata    = 1 - max[{p0:.3f}, {p1:.3f}] = {err_val:.3f}')

print(f'\n{"─"*50}')
print('\n💡 Gözlem: N1 tamamen saf (tüm örnekler aynı sınıfta) → tüm ölçütler = 0')
print('          N3 maksimum safsız (eşit dağılım) → tüm ölçütler maksimum')

---
<a id='8'></a>
## 8. Bilgi Kazancı (Information Gain) ve Kazanç Oranı (Gain Ratio)

$$\\Delta_{info} = I(\\text{parent}) - I(\\text{children})$$

$$\\text{Kazanç Oranı} = \\frac{\\Delta_{info}}{\\text{Bölme Bilgisi}}$$

$$\\text{Bölme Bilgisi} = -\\sum_{j=1}^{k} \\frac{N(v_j)}{N} \\log_2 \\frac{N(v_j)}{N}$$

In [ ]:
def bilgi_kazanci(veri, oznitelik, hedef):
    """Bir öznitelik için bilgi kazancını hesaplar."""
    ust_entropi = hesapla_entropi(veri[hedef])
    agirlikli_entropi = sum(
        len(veri[veri[oznitelik] == v]) / len(veri) *
        hesapla_entropi(veri[veri[oznitelik] == v][hedef])
        for v in veri[oznitelik].unique()
    )
    return ust_entropi - agirlikli_entropi, agirlikli_entropi

def bolme_bilgisi(veri, oznitelik):
    """Bölme bilgisini (split information) hesaplar."""
    bb = 0
    for deger in veri[oznitelik].unique():
        oran = len(veri[veri[oznitelik] == deger]) / len(veri)
        if oran > 0:
            bb -= oran * np.log2(oran)
    return bb

ust_entropi = hesapla_entropi(df_kredit['Temerrut'])

print('📊 Bilgi Kazancı Hesaplama — Kredi Borçlusu Veri Seti')
print('=' * 65)
print(f'  Üst düğüm entropisi I(parent) = {ust_entropi:.3f}')
print(f'  Dağılım: {dict(df_kredit["Temerrut"].value_counts())}\n')

for oznitelik in ['Ev_Sahibi', 'Medeni_Durum']:
    kazanc, agirlikli = bilgi_kazanci(df_kredit, oznitelik, 'Temerrut')
    bb = bolme_bilgisi(df_kredit, oznitelik)
    kazanc_orani = kazanc / bb if bb > 0 else 0
    print(f'  {oznitelik}:')
    print(f'    Ağırlıklı entropi I(children)   = {agirlikli:.3f}')
    print(f'    Bilgi Kazancı Δ                 = {ust_entropi:.3f} - {agirlikli:.3f} = {kazanc:.3f}')
    print(f'    Bölme Bilgisi                   = {bb:.3f}')
    print(f'    Kazanç Oranı                    = {kazanc:.3f} / {bb:.3f} = {kazanc_orani:.3f}')
    print()

In [ ]:
# DÜZELTME: Cinsiyet (Gender) sütunu, sınıfları tam ayırt etmeyecek şekilde düzenlendi.
# Orijinal kod ['M']*10 + ['F']*10 kullanıyordu; bu durum Gender'ı CustomerID kadar
# bilgi kazancına sahip yapıyor ve kazanç oranı karşılaştırmasını anlamsız kılıyordu.
musteri_data = {
    'CustomerID': range(1, 21),
    'Gender': ['M','M','M','F','F','M','F','M','F','F',
               'F','F','F','M','M','F','M','M','M','F'],
    'CarType': ['Family', 'Sports', 'Sports', 'Sports', 'Sports', 'Sports',
                'Sports', 'Sports', 'Sports', 'Luxury',
                'Family', 'Family', 'Family', 'Luxury', 'Luxury',
                'Luxury', 'Luxury', 'Luxury', 'Luxury', 'Luxury'],
    'Class': ['C0']*10 + ['C1']*10
}
df_musteri = pd.DataFrame(musteri_data)

ust_e = hesapla_entropi(df_musteri['Class'])

print('🚗 Kazanç Oranı Örneği: Gender, Car Type, Customer ID')
print('=' * 65)
print(f'  Üst düğüm entropisi = {ust_e:.3f} (10 C0, 10 C1 → eşit dağılım)\n')

for oznitelik, etiket in [('Gender', 'Cinsiyet'), ('CarType', 'Araba Türü')]:
    kazanc, agirlikli = bilgi_kazanci(df_musteri, oznitelik, 'Class')
    bb = bolme_bilgisi(df_musteri, oznitelik)
    kaz_orani = kazanc / bb if bb > 0 else 0
    print(f'  {etiket} ({oznitelik}):')
    print(f'    Çocukların entropisi   = {agirlikli:.3f}')
    print(f'    Bilgi Kazancı          = {ust_e:.3f} - {agirlikli:.3f} = {kazanc:.3f}')
    print(f'    Bölme Bilgisi          = {bb:.3f}')
    print(f'    Kazanç Oranı           = {kazanc:.3f} / {bb:.3f} = {kaz_orani:.3f}')
    print()

kazanc_id = ust_e - 0
bb_id     = np.log2(20)
kaz_orani_id = kazanc_id / bb_id
print(f'  Müşteri Kimliği (CustomerID):')
print(f'    Çocukların entropisi   = 0.000 (her düğümde tek örnek)')
print(f'    Bilgi Kazancı          = {kazanc_id:.3f} (maksimum!)')
print(f'    Bölme Bilgisi          = {bb_id:.3f}')
print(f'    Kazanç Oranı           = {kazanc_id:.3f} / {bb_id:.3f} = {kaz_orani_id:.3f}')
print()
print('💡 Sonuç: Customer ID en yüksek bilgi kazancına sahip olmasına rağmen')
print('         kazanç oranı düşük çünkü çok fazla bölme üretiyor (overfitting riski!)')

In [ ]:
oznitelikler = ['Gender\n(Cinsiyet)', 'CarType\n(Araba Türü)', 'CustomerID\n(Müşteri No)']
bilgi_kazanclari = []
kazanc_oranlari  = []

for oz in ['Gender', 'CarType']:
    k, _ = bilgi_kazanci(df_musteri, oz, 'Class')
    bb   = bolme_bilgisi(df_musteri, oz)
    bilgi_kazanclari.append(k)
    kazanc_oranlari.append(k / bb if bb > 0 else 0)

bilgi_kazanclari.append(kazanc_id)
kazanc_oranlari.append(kaz_orani_id)

x = np.arange(len(oznitelikler))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - w/2, bilgi_kazanclari, w, label='Bilgi Kazancı (ΔInfo)',
               color='#3498db', edgecolor='white', linewidth=1.5)
bars2 = ax.bar(x + w/2, kazanc_oranlari,  w, label='Kazanç Oranı',
               color='#e74c3c', edgecolor='white', linewidth=1.5)

ax.bar_label(bars1, fmt='%.3f', fontsize=10, padding=3, fontweight='bold')
ax.bar_label(bars2, fmt='%.3f', fontsize=10, padding=3, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(oznitelikler, fontsize=11)
ax.set_ylabel('Değer', fontsize=12)
ax.set_title('Bilgi Kazancı vs Kazanç Oranı\nNeden CustomerID ile bölmemeli?',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 1.1)

ax.annotate('Customer ID: Yüksek kazanç\nama düşük kazanç oranı!\n→ Overfitting riski',
            xy=(2, kaz_orani_id), xytext=(1.35, 0.5),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=9, bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

plt.tight_layout()
plt.show()

---
<a id='9'></a>
## 9. Overfitting ve Underfitting

- **Underfitting:** Model çok basit → hem eğitim hem test hatası yüksek
- **Overfitting:** Model eğitimi ezberliyor → eğitim hatası sıfıra yakın ama test hatası yüksek
- **İdeal Model:** Dengeli ağaç derinliği → düşük eğitim ve test hatası

In [ ]:
from sklearn.datasets import make_classification

np.random.seed(42)
X_ovf, y_ovf = make_classification(n_samples=300, n_features=10,
                                     n_informative=5, n_redundant=2,
                                     random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_ovf, y_ovf, test_size=0.33, random_state=42)

derinlikler = range(1, 20)
egitim_hatalari = []
test_hatalari   = []

for d in derinlikler:
    clf_d = DecisionTreeClassifier(max_depth=d, random_state=42)
    clf_d.fit(X_tr, y_tr)
    egitim_hatalari.append(1 - clf_d.score(X_tr, y_tr))
    test_hatalari.append(1 - clf_d.score(X_te, y_te))

ideal_derinlik = list(derinlikler)[np.argmin(test_hatalari)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(derinlikler), egitim_hatalari, 'b-o', linewidth=2,
             markersize=5, label='Eğitim Hatası')
axes[0].plot(list(derinlikler), test_hatalari, 'r-s', linewidth=2,
             markersize=5, label='Test Hatası')
axes[0].axvline(x=ideal_derinlik, color='green', linestyle='--',
                linewidth=2, label=f'İdeal Derinlik={ideal_derinlik}')
axes[0].fill_betweenx([0, max(test_hatalari)+0.05],
                       1, 2.5, alpha=0.1, color='orange', label='Underfitting')
axes[0].fill_betweenx([0, max(test_hatalari)+0.05],
                       12, 20, alpha=0.1, color='red', label='Overfitting')
axes[0].set_xlabel('Ağaç Derinliği', fontsize=11)
axes[0].set_ylabel('Hata Oranı', fontsize=11)
axes[0].set_title('Ağaç Derinliği vs Hata Oranı', fontweight='bold', fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].axis('off')
tablo_veri = [
    ['Underfitting', 'Çok Küçük (1-2)', 'Yüksek', 'Yüksek'],
    ['İdeal Model',  f'Dengeli ({ideal_derinlik} civarı)', 'Düşük', 'Düşük'],
    ['Overfitting',  'Çok Büyük (15+)', '≈ 0', 'Yükseliyor'],
]
col_labels = ['Durum', 'Ağaç Boyutu', 'Eğitim Hatası', 'Test Hatası']
tablo = axes[1].table(cellText=tablo_veri, colLabels=col_labels,
                       loc='center', cellLoc='center')
tablo.auto_set_font_size(False)
tablo.set_fontsize(11)
tablo.scale(1.2, 2.2)
for (row, col), cell in tablo.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2c3e50')
        cell.set_text_props(color='white', fontweight='bold')
    elif row == 1:
        cell.set_facecolor('#fdebd0')
    elif row == 2:
        cell.set_facecolor('#d5f5e3')
    else:
        cell.set_facecolor('#fce4e4')
axes[1].set_title('Underfitting — İdeal — Overfitting', fontweight='bold', fontsize=12)

plt.suptitle('Model Karmaşıklığı ve Genelleme', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\n📊 Optimal derinlik: {ideal_derinlik}')
print(f'   Eğitim hatası: {egitim_hatalari[ideal_derinlik-1]:.3f}')
print(f'   Test hatası  : {test_hatalari[ideal_derinlik-1]:.3f}')

In [ ]:
print('📐 Karmaşıklık Tahmini: errgen(T) = err(T) + Ω × (k / N_train)')
print('=' * 65)
print('Sunumdaki örnek: TL (7 yaprak) vs TR (4 yaprak), N=24 eğitim örneği')
print()

N = 24
kL, kR = 7, 4
errL, errR = 4/N, 6/N

for omega in [0.5, 1.0, 0.25]:
    errgen_L = errL + omega * (kL / N)
    errgen_R = errR + omega * (kR / N)
    tercih = 'TL' if errgen_L < errgen_R else 'TR'
    print(f'  Ω = {omega}:')
    print(f'    errgen(TL) = {errL:.3f} + {omega} × ({kL}/{N}) = {errgen_L:.4f}')
    print(f'    errgen(TR) = {errR:.3f} + {omega} × ({kR}/{N}) = {errgen_R:.4f}')
    print(f'    → Tercih edilen model: {tercih}')
    print()

omega_vals     = np.linspace(0, 2, 200)
errgen_L_vals  = errL + omega_vals * (kL / N)
errgen_R_vals  = errR + omega_vals * (kR / N)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(omega_vals, errgen_L_vals, 'b-', linewidth=2.5, label=f'TL (k={kL} yaprak)')
ax.plot(omega_vals, errgen_R_vals, 'r-', linewidth=2.5, label=f'TR (k={kR} yaprak)')

omega_kesisim = (errR - errL) / ((kL - kR) / N)
ax.axvline(x=omega_kesisim, color='green', linestyle='--', linewidth=2,
           label=f'Kesişim Ω = {omega_kesisim:.2f}')
ax.scatter([omega_kesisim], [errL + omega_kesisim * (kL/N)], s=150, zorder=5, color='green')

y_min = min(errgen_L_vals.min(), errgen_R_vals.min())
y_max = max(errgen_L_vals.max(), errgen_R_vals.max())
ax.fill_betweenx([y_min, y_max], 0, omega_kesisim,
                  alpha=0.1, color='blue', label='TL tercih edilir')
ax.fill_betweenx([y_min, y_max], omega_kesisim, 2,
                  alpha=0.1, color='red', label='TR tercih edilir')

ax.set_xlabel('Ω (Hiperparametre)', fontsize=12)
ax.set_ylabel('Genelleme Hata Tahmini', fontsize=12)
ax.set_title('Ω Değerine Göre Model Tercihi Değişimi', fontweight='bold', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 2)
plt.tight_layout()
plt.show()

---
<a id='10'></a>
## 10. Model Seçimi ve Cross-Validation

**Holdout Yöntemi:** Veriyi eğitim ve test olarak ikiye böl.

**Cross-Validation:** Veriyi k parçaya böl, her seferinde farklı parçayı test için kullan.

In [ ]:
from sklearn.datasets import make_classification

np.random.seed(42)
X_all, y_all = make_classification(n_samples=500, n_features=10,
                                    n_informative=5, random_state=42)

X_Dtrain, X_Dtest, y_Dtrain, y_Dtest = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42)

X_Dtr, X_Dval, y_Dtr, y_Dval = train_test_split(
    X_Dtrain, y_Dtrain, test_size=0.33, random_state=42)

print('📊 Veri Bölme Özeti')
print('=' * 45)
print(f'  Toplam veri (D)       : {len(X_all)} örnek')
print(f'  D.train               : {len(X_Dtrain)} örnek')
print(f'    D.tr (eğitim)       : {len(X_Dtr)} örnek')
print(f'    D.val (doğrulama)   : {len(X_Dval)} örnek')
print(f'  D.test (final test)   : {len(X_Dtest)} örnek')
print()

derinlik_aralik = range(1, 15)
dogrulama_hatalari = []

for d in derinlik_aralik:
    clf_d = DecisionTreeClassifier(max_depth=d, random_state=42)
    clf_d.fit(X_Dtr, y_Dtr)
    dogrulama_hatalari.append(1 - clf_d.score(X_Dval, y_Dval))

optimal_p = list(derinlik_aralik)[np.argmin(dogrulama_hatalari)]
print(f'✅ Optimal hiperparametre p* = max_depth={optimal_p}')
print(f'   Doğrulama hatası errval(p*) = {min(dogrulama_hatalari):.3f}')

clf_final = DecisionTreeClassifier(max_depth=optimal_p, random_state=42)
clf_final.fit(X_Dtrain, y_Dtrain)
test_hatasi = 1 - clf_final.score(X_Dtest, y_Dtest)
print(f'   Final test hatası errtest    = {test_hatasi:.3f}')

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(list(derinlik_aralik), dogrulama_hatalari, 'r-o',
        linewidth=2, markersize=6, label='Doğrulama Hatası errval(p)')
ax.axvline(x=optimal_p, color='green', linestyle='--', linewidth=2.5,
           label=f'p* = {optimal_p} (optimal)')
ax.scatter([optimal_p], [min(dogrulama_hatalari)], s=200, zorder=5,
           color='green', label=f'Min errval = {min(dogrulama_hatalari):.3f}')
ax.set_xlabel('Hiperparametre p (max_depth)', fontsize=12)
ax.set_ylabel('Doğrulama Hata Oranı', fontsize=12)
ax.set_title('Hiperparametre Seçimi: Doğrulama Kümesi ile Optimal Derinlik',
             fontweight='bold', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# DÜZELTİLDİ: f-string içinde kaçırılmış tırnak (escaped quote) kullanan satır,
# Python 3.10'da SyntaxError veriyordu. Çözüm: ara değişken kullanımı.
print('🔄 Cross-Validation (Çapraz Doğrulama) — 3-Fold Örneği')
print('=' * 60)
print('Veri 3 eşit parçaya (S1, S2, S3) bölünüyor:\n')

kf = KFold(n_splits=3, shuffle=True, random_state=42)
fold_hatalari = []
tum_foldlar = {1, 2, 3}

for fold, (train_idx, test_idx) in enumerate(kf.split(X_Dtrain), 1):
    X_fold_tr = X_Dtrain[train_idx]
    X_fold_te = X_Dtrain[test_idx]
    y_fold_tr = y_Dtrain[train_idx]
    y_fold_te = y_Dtrain[test_idx]

    clf_fold = DecisionTreeClassifier(max_depth=optimal_p, random_state=42)
    clf_fold.fit(X_fold_tr, y_fold_tr)
    hata = 1 - clf_fold.score(X_fold_te, y_fold_te)
    fold_hatalari.append(hata)

    # FIX: ara değişken ile f-string sorununu önle
    diger_foldlar = sorted(tum_foldlar - {fold})
    egitim_str = 'S' + '+S'.join(str(f) for f in diger_foldlar)
    print(f'  Run {fold}: Eğitim={egitim_str}, Test=S{fold} → Hata: {hata:.3f}')

ortalama_hata = np.mean(fold_hatalari)
std_hata      = np.std(fold_hatalari)
print(f'\n  📊 Ortalama CV Hatası: {ortalama_hata:.3f} ± {std_hata:.3f}')

cv_skorlar = cross_val_score(
    DecisionTreeClassifier(max_depth=optimal_p, random_state=42),
    X_Dtrain, y_Dtrain, cv=5
)
print(f'\n  🔁 5-Fold CV Doğrulukları: {cv_skorlar.round(3)}')
print(f'  📊 Ortalama: {cv_skorlar.mean():.3f} ± {cv_skorlar.std():.3f}')

---
<a id='11'></a>
## 11. Model Karşılaştırması — İstatistiksel Test

İki model karşılaştırılırken:
$$\\sigma^2_d \\approx \\frac{e_A(1-e_A)}{n_A} + \\frac{e_B(1-e_B)}{n_B}$$

$$d_t = d \\pm z_{\\alpha/2} \\hat{\\sigma}_d$$

Güven aralığı **0'ı kapsıyorsa** → fark istatistiksel olarak anlamsız.

In [ ]:
from scipy import stats

def model_karsilastirma(eA, nA, eB, nB, guven=0.95):
    """
    İki modeli istatistiksel olarak karşılaştırır.
    eA, eB: hata oranları | nA, nB: test kümesi boyutları
    """
    d        = eA - eB
    sigma2_d = (eA * (1 - eA) / nA) + (eB * (1 - eB) / nB)
    sigma_d  = np.sqrt(sigma2_d)
    alpha    = 1 - guven
    z        = stats.norm.ppf(1 - alpha / 2)
    alt_sinir = d - z * sigma_d
    ust_sinir = d + z * sigma_d
    anlamli   = not (alt_sinir <= 0 <= ust_sinir)
    return {'d': d, 'sigma_d': sigma_d, 'z': z,
            'alt': alt_sinir, 'ust': ust_sinir, 'anlamli': anlamli}

# MA: %85 doğruluk → hata oranı eA=0.15 | MB: %75 doğruluk → hata oranı eB=0.25
print('📊 Sunumdaki Model Karşılaştırma Örneği')
print('=' * 55)
print('  MA: %85 doğruluk, 30 test örneği  → eA = 0.15')
print('  MB: %75 doğruluk, 5000 test örneği → eB = 0.25')
print()

sonuc = model_karsilastirma(eA=0.15, nA=30, eB=0.25, nB=5000)

# DÜZELTİLDİ: d = eA - eB negatif çıkar (MA daha iyi), mutlak değer gösterimi eklendi
print(f'  d = eA - eB = 0.15 - 0.25 = {sonuc["d"]:.3f}')
print(f'  |d| = {abs(sonuc["d"]):.3f}  (MA, MB\'den bu kadar daha iyi)')
print(f'  σ²_d = 0.15×0.85/30 + 0.25×0.75/5000 = {sonuc["sigma_d"]**2:.4f}')
print(f'  σ_d  = {sonuc["sigma_d"]:.4f}')
print(f'  z_{{α/2}} = {sonuc["z"]:.3f}  (%95 güven için)')
print(f'  dt = {sonuc["d"]:.3f} ± {sonuc["z"]:.3f} × {sonuc["sigma_d"]:.4f}')
print(f'     = {sonuc["d"]:.3f} ± {sonuc["z"]*sonuc["sigma_d"]:.4f}')
print(f'  Güven Aralığı: [{sonuc["alt"]:.3f}, {sonuc["ust"]:.3f}]')
print()
if sonuc['anlamli']:
    print('  ✅ Sonuç: Fark istatistiksel olarak ANLAMLIDIR')
else:
    print('  ❌ Sonuç: Fark istatistiksel olarak ANLAMSIZDIR')
    print('  Güven aralığı 0 değerini kapsıyor → MA\'nın gerçekten daha iyi')
    print('  olduğunu söyleyemeyiz! Neden? MA yalnızca 30 örnekle test edildi')
    print('  → çok yüksek belirsizlik (geniş güven aralığı).')

In [ ]:
print('\n📐 Örneklem Boyutunun Güven Aralığına Etkisi')
print('=' * 55)
print('MA\'nın %85 doğruluğu sabit, test kümesi büyüklüğü değişiyor:\n')

boyutlar = [30, 100, 500, 1000, 5000]
for n in boyutlar:
    s = model_karsilastirma(eA=0.15, nA=n, eB=0.25, nB=5000)
    genislik = s['ust'] - s['alt']
    etiket = '✅ Anlamlı' if s['anlamli'] else '❌ Anlamsız'
    print(f'  n={n:5d}: GA=[{s["alt"]:+.3f}, {s["ust"]:+.3f}]  '
          f'Genişlik={genislik:.3f}  {etiket}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, n in enumerate(boyutlar):
    s = model_karsilastirma(eA=0.15, nA=n, eB=0.25, nB=5000)
    renk = '#2ecc71' if s['anlamli'] else '#e74c3c'
    axes[0].errorbar(i, s['d'], yerr=s['z']*s['sigma_d'],
                     fmt='o', color=renk, capsize=8, capthick=2,
                     markersize=8, linewidth=2)
    axes[0].text(i, s['d'] + s['z']*s['sigma_d'] + 0.01,
                '✅' if s['anlamli'] else '❌', ha='center', fontsize=14)

axes[0].axhline(y=0, color='black', linestyle='--', linewidth=2, label='d=0 (fark yok)')
axes[0].set_xticks(range(len(boyutlar)))
axes[0].set_xticklabels([f'n={n}' for n in boyutlar], fontsize=10)
axes[0].set_ylabel('Hata Farkı (d = eA − eB)', fontsize=11)
axes[0].set_title('Örneklem Boyutu ve Güven Aralığı\n(%95 Güven Düzeyi)',
                   fontweight='bold', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

genislikler = []
n_aralik = range(10, 2000, 20)
for n in n_aralik:
    s = model_karsilastirma(eA=0.15, nA=n, eB=0.25, nB=5000)
    genislikler.append(s['ust'] - s['alt'])

axes[1].plot(list(n_aralik), genislikler, 'b-', linewidth=2)
axes[1].set_xlabel('MA Test Kümesi Boyutu (nA)', fontsize=11)
axes[1].set_ylabel('Güven Aralığı Genişliği', fontsize=11)
axes[1].set_title('Örneklem Büyüdükçe Belirsizlik Azalır',
                   fontweight='bold', fontsize=12)
axes[1].grid(True, alpha=0.3)
axes[1].fill_between(list(n_aralik), genislikler, alpha=0.2, color='blue')

plt.suptitle('İstatistiksel Model Karşılaştırması', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 📚 Özet ve Temel Kavramlar

| Kavram | Açıklama |
|--------|----------|
| **Sınıflandırma** | (x, y) çiftlerinden model öğrenme ve yeni x için y tahmin etme |
| **Tümevarım** | Eğitim verisinden model oluşturma (induction) |
| **Tümdengelim** | Öğrenilen modeli test verisine uygulama (deduction) |
| **Confusion Matrix** | f₁₁, f₁₀, f₀₁, f₀₀ değerlerinden oluşan performans özeti |
| **Entropi** | Düğümdeki belirsizliği ölçer; 0=tam saf, max=eşit dağılım |
| **Gini** | Yanlış sınıflandırma olasılığı; 0=tam saf |
| **Bilgi Kazancı** | Üst düğüm − ağırlıklı çocuk entropisi |
| **Kazanç Oranı** | Bilgi kazancını bölme sayısına göre normalize eder |
| **Overfitting** | Model ezberleme: eğitim hatası↓ test hatası↑ |
| **Underfitting** | Model çok basit: her ikisi de↑ |
| **Cross-Validation** | Genelleme performansını güvenilir şekilde ölçme yöntemi |
| **Güven Aralığı** | İki modelin karşılaştırılmasında istatistiksel anlamlılık testi |

---
> 💡 **Bir Sonraki Derste:** Diğer sınıflandırma algoritmaları (k-NN, Naive Bayes, SVM, vb.)